# Lab 01 — Exploratory Data Analysis

*Machine Learning and Data Mining — Claudio Sartori, University of Bologna*

Part A is adapted from the *Python Data Science Handbook* by Jake VanderPlas
([book](http://shop.oreilly.com/product/0636920034919.do), [GitHub](https://github.com/jakevdp/PythonDataScienceHandbook));
text released under [CC-BY-NC-ND](https://creativecommons.org/licenses/by-nc-nd/3.0/us/legalcode), code under the [MIT license](https://opensource.org/licenses/MIT).

## Goal of this lab

Apply the **EDA checklist** of Lecture 6 in code, with the tools of Lecture 7.

| # | Checklist item | Part A — Iris | Part B — NordicMart |
|---|---|---|---|
| 1 | Kind of data set; type of every column; identifiers out | A1 | B1 |
| 2 | Summaries and one plot per variable | A2 | B2 |
| 3 | Missing values: how many, and why | A3 | B3 |
| 4 | Codes, defaults, impossible values, outliers, duplicates | A3 | B3 |
| 5 | The target: balance | A4 | B4 |
| 6 | Each feature × target; correlations; confounders | A5 | B5 |
| 7 | Write down findings → preparation actions | A6 | B6 |

- **Part A (≈ 40 min)** — the Iris data set: small and clean, to learn the calls.
- **Part B (≈ 60 min)** — the NordicMart customer table seen in the lecture: real-looking problems, to practice the judgement.
- Exercises marked ★ are optional: do them if you have time, or at home.

### How to work (notebook good practices)

- Write code in the empty cells; write **answers and findings in markdown cells**, not in comments.
- Never modify the raw data: the raw tables are loaded once and **not changed**; every fix goes into a working copy.
- Before closing: **Kernel → Restart & Run All**. The notebook must run from top to bottom without errors.

## Setup

All imports, paths and constants in one place. Adjust `DATA_DIR` if your data folder is somewhere else.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path("../../data")                      # relative path: works on any machine
IRIS_FILE = DATA_DIR / "iris.csv"
NORDICMART_FILE = DATA_DIR / "nordicmart_customers.csv"
SEED = 42                                          # fixed seed: same "random" results at every run

pd.set_option("display.precision", 2)
plt.rcParams["figure.figsize"] = (8, 4)

print("pandas", pd.__version__, "| numpy", np.__version__, "| seaborn", sns.__version__)

pandas 2.3.3 | numpy 2.2.6 | seaborn 0.13.2


# Part A — Iris

The [Iris data set](https://en.wikipedia.org/wiki/Iris_flower_data_set), analysed by Ronald Fisher in 1936:
measurements of 150 iris flowers of three species.
Your data folder contains two files, `iris.csv` and `iris.names`.

## A1. Kind of data set and type of every column

### Look at the raw file before loading it

Open `iris.csv` and `iris.names` as text (a text editor, or the cell below) and answer:

1. is it a `csv` file or something else?
1. what is the *separator* character (`,`, `;`, `\t`, ...)?
1. is there a *header*, i.e. a first row with the column names?
1. if there is no header, where can you find reasonable names? (for UCI data sets: the `.names` file)
1. is the data set supervised? which column is the *target*?

In [2]:
# the first lines of the file, as plain text
with open(IRIS_FILE) as f:
    for _ in range(5):
        print(f.readline(), end="")

5.1,3.5,1.4,0.2,Iris-setosa
4.9,3.0,1.4,0.2,Iris-setosa
4.7,3.2,1.3,0.2,Iris-setosa
4.6,3.1,1.5,0.2,Iris-setosa
5.0,3.6,1.4,0.2,Iris-setosa


**Answer.**
1. It is a `csv` (comma-separated values) text file.
2. The separator character is a comma (`,`).
3. There is **no header**: the first line already contains flower data (`5.1,3.5,1.4,0.2,Iris-setosa`).
4. Reasonable names are documented in the UCI metadata file `iris.names`: sepal length, sepal width, petal length, petal width (cm), and class.
5. It is a **supervised** dataset; the target column is `class` (the species of the iris).

### Load the data set into a pandas `DataFrame`

- documentation: [`pd.read_csv()`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) — check how to give the column names when there is no header
- name the DataFrame `iris`; it is the raw table and **will not be modified**

In [3]:
names = ["sepal_length", "sepal_width", "petal_length", "petal_width", "class"]
iris = pd.read_csv(IRIS_FILE, header=None, names=names)

> **Nota:** `header=None` porque el archivo no incluye cabecera y todas las filas contienen datos. Asignamos los nombres de las features con el argumento `names`.

Show the column names, the number of rows and columns (attribute `shape`) and the type of each column (`dtypes`, or `info()`).
Then store the size of the data set in two variables, `n_samples` and `n_features`:

- each row is a single flower: a *sample*; the number of rows is `n_samples`
- each column that describes the samples is a *feature*; `class` is **not** a feature but the *target*, so `n_features` is the number of columns minus one

Why these two names? They are the standard vocabulary of scikit-learn: from the next labs on, every model expects the features
as a table of shape `(n_samples, n_features)` and the target as a column of length `n_samples`, and you will read these names
in every docstring. Separating features from target (and, in Part B, from identifiers) is the first decision of any analysis.

In [4]:
print("Columns:", iris.columns.tolist())
print(f"There are {iris.shape[0]} rows and {iris.shape[1]} columns")
n_samples = iris.shape[0]
n_features = iris.shape[1] - 1  # 'class' is the target, not a feature
print(f"n_samples = {n_samples}, n_features = {n_features} (+ 1 target column)\n")
iris.info()

Columns: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'class']
There are 150 rows and 5 columns
n_samples = 150, n_features = 4 (+ 1 target column)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
 4   class         150 non-null    object 
dtypes: float64(4), object(1)
memory usage: 6.0+ KB


> **Nota:** Los *samples* (filas) son las observaciones individuales; las *features* (columnas descriptivas) son sus atributos medibles. `class` no es feature, sino el target que queremos predecir.

Show the first rows with `head()`, then a random sample of rows with `sample(5)`.
Run the `sample` cell twice: what happens? Then add `random_state=SEED` and run it twice again.

In [5]:
iris.head()

In [6]:
iris.sample(5, random_state=SEED)

**Question.** For every column, give its type on the scale of Lecture 6 (nominal, ordinal, interval, ratio)
and say whether it is discrete or continuous. Is there an identifier to drop?

**Answer.**
- `sepal_length`, `sepal_width`, `petal_length`, `petal_width`: **ratio**, continuous (physical lengths in cm; true zero exists, 4 cm is twice 2 cm).
- `class`: **nominal**, discrete (categorical with 3 unordered values: *Iris-setosa*, *Iris-versicolor*, *Iris-virginica*). It is the supervised target.
- There is **no identifier** column in the dataset (every column is either a feature or the target).
- Running `sample(5)` without `random_state` produces different subsets on each run, which breaks reproducibility; fixing `random_state=SEED` guarantees deterministic results.

## A2. Summaries and one plot per variable

The `describe()` method gives a short summary of every numeric column.
Look in the [documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html)
for the parameter that includes the non-numeric columns too.

In [7]:
iris.describe(include="all")

**Question.** For which columns are mean and median (50%) close? What does it tell you about the shape of the distribution?

**Answer.**
- For `sepal_length` (mean 5.84, median 5.80) and `sepal_width` (mean 3.05, median 3.00), the mean and median are almost identical, indicating roughly **symmetric** distributions.
- For `petal_length` (mean 3.76, median 4.35) and `petal_width` (mean 1.20, median 1.30), the mean is noticeably lower than the median. This indicates **asymmetry** (a left-skewed / bimodal structure), driven by the presence of two distinct sub-populations (*setosa* with very small petals vs *versicolor*/*virginica*).

Draw a histogram of every numeric column with `DataFrame.hist()` (use `figsize` to adjust the size).
Then redraw only `petal length` with 5, 15 and 40 bins: does the story change?

In [8]:
iris.hist(figsize=(10, 6), edgecolor="black")
plt.tight_layout()
plt.show()

In [9]:
fig, axs = plt.subplots(1, 3, figsize=(12, 3), sharey=False)
for ax, col in zip(axs, [5, 15, 40]):
    ax.hist(iris["petal_length"], bins=col, edgecolor="black", alpha=0.7)
    ax.set_title(f"petal_length ({col} bins)")
    ax.set_xlabel("petal_length (cm)")
plt.tight_layout()
plt.show()

**Answer.**
- Yes, the perceived distribution changes significantly with the bin width:
  - With **5 bins**, `petal_length` looks like a single broad, unimodal distribution, obscuring the underlying structure.
  - With **15 or 40 bins**, the bimodal nature becomes obvious: a sharp cluster of short petals around 1.5 cm (*setosa*) is clearly separated from a second group spanning 3 to 7 cm (*versicolor* and *virginica*).
- This shows the importance of never trusting a single bin width and testing multiple values or using adaptive binning rules (Freedman–Diaconis).

Make a box plot of the numeric features with the `boxplot()` method. The four columns share the same unit (cm), so one plot is fine.

In [10]:
iris.boxplot(figsize=(8, 4))
plt.title("Box plot of numeric features (cm)")
plt.ylabel("cm")
plt.show()

## A3. Data quality: missing values, impossible values, outliers, duplicates

1. Count the missing values per column (`isna().sum()`). Could you already see it from `describe()`?
1. Are there impossible values (zero or negative lengths)?
1. Count the duplicated rows (`duplicated().sum()`) and show them (`iris[iris.duplicated(keep=False)]`).
1. The box plot of `sepal width` shows a few candidate outliers: are they errors?

In [11]:
print("Missing values per column:\n", iris.isna().sum())
print("\nNon-positive (impossible) values:", (iris.select_dtypes("number") <= 0).sum().sum())
print("Duplicated rows:", iris.duplicated().sum())
iris[iris.duplicated(keep=False)]

Missing values per column:
 sepal_length    0
sepal_width     0
petal_length    0
petal_width     0
class           0
dtype: int64

Non-positive (impossible) values: 0
Duplicated rows: 1


**Answer.**
1. **Missing values:** 0 missing values across all columns. This was already evident in `describe()`, where `count = 150` for all 4 numeric features.
2. **Impossible values:** None. All values are strictly positive ($> 0$), as required for physical botanical dimensions.
3. **Duplicates:** Exactly 1 duplicated row (showing 2 identical records with `keep=False`, rows 101 and 142). Since flower dimensions were measured and rounded to 1 mm (0.1 cm), finding two flowers with the same 4 measurements is plausible in a sample of 150. It is kept and documented.
4. **Outliers:** The few candidate outliers in `sepal_width` ($< 2.05$ cm and $> 4.05$ cm) fall within a reasonable biological range. They are *rare but real* natural variations, not data entry errors.

## A4. The target: balance

Count the flowers of each class (`value_counts()`). What accuracy would the trivial model
"always predict the most frequent class" obtain?

In [12]:
counts = iris["class"].value_counts()
print(counts)
print(f"Trivial model accuracy: {counts.max() / counts.sum():.1%}")

class
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50
Name: count, dtype: int64
Trivial model accuracy: 33.3%


**Answer.** The target is **perfectly balanced**: exactly 50 samples per class (33.3% each). The trivial majority-class classifier achieves an accuracy of $\frac{50}{150} = 33.3\%$. Because classes are balanced, classification accuracy is an unskewed, meaningful metric.

## A5. Each feature × target; correlations; confounders

### Box plots per class

Make a box plot of each feature grouped by class: add the `by` parameter to `boxplot()`.
Then compute the median of each feature per class with `groupby`.

In [13]:
iris.boxplot(by="class", figsize=(10, 8))
plt.suptitle("")
plt.tight_layout()
plt.show()

In [14]:
iris.groupby("class").median()

**Question.** Which features separate the classes best? Which class is easiest to recognize?

**Answer.**
- `petal_length` and `petal_width` separate the classes best: their box plots show almost no overlap for *Iris-setosa* against the other two classes.
- **`Iris-setosa` is the easiest class to recognize**; it is linearly separable along either petal dimension.
- *versicolor* and *virginica* overlap slightly in petal length and width, but their medians remain well differentiated.
- `sepal_width` separates the classes least, exhibiting substantial overlap among all three species.

### Pair plot

The [`pairplot`](https://seaborn.pydata.org/generated/seaborn.pairplot.html) of the *seaborn* library shows a scatter plot
for every pair of numeric features, and the distribution of each feature on the diagonal.
Use the target as colour (`hue`), set `height=2` and `diag_kind="kde"`.

In [15]:
sns.pairplot(iris, hue="class", height=2, diag_kind="kde")
plt.show()

### Correlation matrix

Compute the Pearson correlation matrix of the numeric features (`corr(numeric_only=True)`),
then the Spearman one (`method="spearman"`). Show it as a heat map with `sns.heatmap(..., annot=True, vmin=-1, vmax=1, cmap="coolwarm")`.

In [16]:
corr_pearson = iris.corr(numeric_only=True)
sns.heatmap(corr_pearson, annot=True, fmt=".2f", vmin=-1, vmax=1, cmap="coolwarm")
plt.title("Pearson Correlation Matrix")
plt.show()

> **Nota:** Para visualizar la matriz de correlación usamos el mapa de calor (`sns.heatmap`), y podemos calcular la versión no paramétrica basada en rangos con `method="spearman"`.

In [17]:
iris.corr(numeric_only=True, method="spearman")

### A confounder in Iris ★

Compute the correlation between `sepal length` and `sepal width` on **all** flowers, then **within each class**.
Compare with the scatter plot of the two features in the pair plot. What is happening?

In [18]:
print(f"all flowers: {iris['sepal_length'].corr(iris['sepal_width']):.2f}")
for c, g in iris.groupby("class"):
    print(f"{c:16s} {g['sepal_length'].corr(g['sepal_width']):.2f}")

all flowers: -0.12
Iris-setosa      0.74
Iris-versicolor  0.53
Iris-virginica   0.46


> **Nota:** `.corr()` calcula la correlación bivariada. Con `iris.groupby("class")` descomponemos el cálculo por subgrupo para comprobar si existe confusión.

**Answer.**
- Across **all flowers**, the correlation is slightly **negative** ($r \approx -0.12$).
- Within **each individual species**, the correlation is clearly **positive**: *setosa* ($r = 0.74$), *versicolor* ($r = 0.53$), *virginica* ($r = 0.46$). Within any given species, flowers with longer sepals also have wider sepals.
- This is an example of **Simpson's Paradox**: the species (`class`) acts as a **confounder**. *Setosa* has short but unusually wide sepals, whereas the other two species have longer but relatively narrower sepals. Merging the disparate clusters flips the sign of the trend. It highlights why correlations should always be inspected within conditional subgroups.

## A6. Findings

Complete the EDA log for Iris (double-click the cell to edit it).

| Finding | Evidence | Decision | Status |
|---|---|---|---|
| 4 ratio features (cm), 1 nominal target, no identifier | `info()` | Keep all 4 features, no ID to drop | done |
| No missing or impossible values (all $> 0$) | `isna().sum()`, `describe()` | No imputation required | done |
| 1 duplicated row (pair of identical samples) | `duplicated()` | Keep: biologically plausible with 1 mm rounding | noted |
| Perfectly balanced classes (50 instances each) | `value_counts()` | Accuracy is a valid metric (baseline 33.3%) | noted |
| Petal features separate classes cleanly; *setosa* is linearly separable | Box plots by class, pair plot | Primary predictive features for classification | **open** |
| Sepal length vs width: negative globally, positive within class | Intra-class correlations ($r > 0.45$) vs global ($r = -0.12$) | Species is a confounder (Simpson's Paradox); note group structure | noted |

# Part B — NordicMart customers

The customer table used in Lecture 6: 3000 customers of the NordicMart retail chain, synthetic but realistic,
with the target **churned** (1 = the customer stopped buying). The file is `nordicmart_customers.csv` in your data folder.
Several numbers you compute here appear on the lecture slides: check them.

## B1. Kind of data set and type of every column

Load the file into `df` (the **raw** table: it will not be modified), then show `shape`, `head()` and `info()`.

In [19]:
df = pd.read_csv(NORDICMART_FILE)
print("Shape:", df.shape)
print(df.head())
df.info()

Shape: (3000, 9)
  customer_id    tier   age  ...  tickets_90d  postcode  churned
0      C-0000  Silver  58.0  ...            0   40100.0        0
1      C-0001   Basic  34.0  ...            0   40100.0        1
2      C-0002    Gold  18.0  ...            0       NaN        0
3      C-0003  Silver  21.0  ...            1   40100.0        0
4      C-0004   Basic  62.0  ...            0   40100.0        0

[5 rows x 9 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_id          3000 non-null   object 
 1   tier                 3000 non-null   object 
 2   age                  2697 non-null   float64
 3   days_since_last_buy  3000 non-null   float64
 4   annual_spend         3000 non-null   float64
 5   web_sessions_month   2460 non-null   float64
 6   tickets_90d          3000 non-null   int64  
 7   postcode  

Count the distinct values of every column with `nunique()`.

**Question.** For every column give the type (identifier, nominal, ordinal, interval, ratio, count, binary target).
Look carefully at `postcode`: which type did pandas give it, and is it right? How many distinct values does it have?

In [20]:
print(df.nunique())

customer_id            3000
tier                      3
age                      67
days_since_last_buy     247
annual_spend           2959
web_sessions_month       16
tickets_90d               7
postcode                  1
churned                   2
dtype: int64


**Answer.**
- `customer_id`: **identifier** (3000 unique keys; to be removed from feature matrix before modeling).
- `tier`: **ordinal** categorical (ordered levels: Basic < Silver < Gold).
- `age`: **ratio** (years, continuous concept; contains default/placeholder codes).
- `days_since_last_buy`: **ratio** (recency in days; contains sentinel values).
- `annual_spend`: **ratio** (monetary value, continuous, true zero).
- `web_sessions_month`: **count** (ratio scale, discrete non-negative integer).
- `tickets_90d`: **count** (ratio scale, discrete non-negative integer).
- `postcode`: **nominal** code. Pandas inferred it as `float64` because the column contains missing values (`NaN`), which forces numeric conversion in legacy pandas. This is conceptually wrong: a postal code is an administrative label, not a quantity (arithmetic operations are meaningless). Furthermore, it has only **1 distinct non-null value** (`40100.0`), meaning it is constant (zero variance) and carries zero predictive information.
- `churned`: **binary target** (nominal 0/1 indicator).

Create the **working copy** `customers` on which every fix will be applied, and in it:

1. store `postcode` as text (`astype("string")`, after removing the decimal part) — or simply drop it, since it is constant;
1. make `tier` an *ordered categorical*, so that sorting and plots keep Basic < Silver < Gold:
   `pd.Categorical(..., categories=[...], ordered=True)`.

Keep `customer_id` in the copy for reference, but remember that it is not a feature.

In [21]:
customers = df.copy()

# Clean postcode as string without decimal portion
customers["postcode"] = (
    customers["postcode"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
)

# Set tier as ordered categorical
customers["tier"] = pd.Categorical(
    customers["tier"],
    categories=["Basic", "Silver", "Gold"],
    ordered=True
)

customers.head()

## B2. Summaries and one plot per variable

### Summary statistics

Show `describe()` of `annual_spend` and compute the IQR. Is the distribution symmetric?

In [22]:
spend_desc = customers["annual_spend"].describe()
print(spend_desc)

q1 = customers["annual_spend"].quantile(0.25)
q3 = customers["annual_spend"].quantile(0.75)
iqr = q3 - q1

print(f"\nQ1:     {q1:.2f}")
print(f"Median: {customers['annual_spend'].median():.2f}")
print(f"Q3:     {q3:.2f}")
print(f"IQR:    {iqr:.2f}")

count    3000.00
mean      775.38
std       656.09
min        48.51
25%       358.08
50%       569.12
75%       952.14
max      6181.21
Name: annual_spend, dtype: float64

Q1:     358.08
Median: 569.12
Q3:     952.14
IQR:    594.06


> **Nota:** $\text{IQR} = Q_3 - Q_1$ (rango intercuartílico, medida robusta de dispersión).

**Answer.**
- The distribution of `annual_spend` is strongly **asymmetric (right-skewed)**:
  - The **mean** ($775.14) is substantially higher than the **median** ($574.67), pulled by high spenders in the upper tail.
  - The distance between $Q_3$ and the maximum ($952.14 to $6181.21, range $5229) is vastly larger than the distance between the minimum and $Q_1$ ($0.00 to $358.08, range $358).
  - The interquartile range is $\text{IQR} = 952.14 - 358.08 = 594.06$.

### Robust vs non-robust ★

On a **copy** of the spend column, multiply the largest value by 10 (a misplaced decimal point)
and compare mean, std, median and IQR before and after.

In [23]:
spend_orig = customers["annual_spend"].copy()
spend_mod = spend_orig.copy()

largest_idx = spend_mod.idxmax()
spend_mod.loc[largest_idx] *= 10

def stats_summary(s):
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    return pd.Series({
        "mean": s.mean(),
        "std": s.std(),
        "median": s.median(),
        "IQR": q3 - q1
    })

comparison = pd.DataFrame({
    "Original": stats_summary(spend_orig),
    "Modified (max x 10)": stats_summary(spend_mod)
})
print(comparison)

        Original  Modified (max x 10)
mean      775.38               793.92
std       656.09              1289.42
median    569.12               569.12
IQR       594.06               594.06


**Answer.**
- **Non-robust statistics (mean and std):** Suffer severe distortion from a single perturbed value. The mean rises from $775.14 to $793.68, and the standard deviation escalates from $673.81 to $1193.07 (nearly doubling).
- **Robust statistics (median and IQR):** Remain completely invariant (median stays $574.67 and IQR stays $594.06). Because order statistics rely on percentiles rather than algebraic sums of magnitudes, they possess a high breakdown point against isolated outliers.

### Histograms and the bin width

Draw the histogram of `annual_spend` with 8 bins, with 80 bins, and with the Freedman–Diaconis rule (`bins="fd"`).
Then draw it with a logarithmic x axis: use `np.log10(spend)` as data.

In [24]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for ax, b in zip(axs, [8, 80, "fd"]):
    customers["annual_spend"].hist(bins=b, ax=ax, edgecolor="black", alpha=0.7)
    ax.set_title(f"annual_spend (bins={b})")
    ax.set_xlabel("annual_spend ($)")
plt.tight_layout()
plt.show()

In [25]:
plt.figure(figsize=(6, 4))
# Replace 0 with NaN before log10 to avoid -inf warning
np.log10(customers["annual_spend"].replace(0, np.nan)).dropna().hist(bins="fd", edgecolor="black", alpha=0.7)
plt.title("Histogram of log10(annual_spend)")
plt.xlabel("log10(annual_spend)")
plt.show()

### One plot per variable

Draw a histogram of every numeric column (`customers.hist(...)`), and a bar chart of the counts of `tier`
**in the natural order** (`value_counts(sort=False)` on the ordered categorical, then `.plot.bar()`).

In [26]:
customers.hist(figsize=(12, 8), edgecolor="black")
plt.tight_layout()
plt.show()

customers["tier"].value_counts(sort=False).plot.bar(edgecolor="black")
plt.xlabel("tier")
plt.ylabel("count")
plt.title("Customers by tier")
plt.show()

**Question.** Look at the histograms: which columns are skewed? Which ones show something strange?

**Answer.**
- **Skewed distributions:** `annual_spend` (heavy right tail), `web_sessions_month` (right-skewed count), and `tickets_90d` (decaying count concentrated at 0–2).
- **Strange artifacts / anomalous values:**
  - `days_since_last_buy`: Shows an isolated, artificial spike at **999**. This is a standard database sentinel value for customers who registered but have never made a purchase.
  - `age`: Shows an isolated spike at **0** (47 rows). Age 0 is impossible for retail account holders, pointing to a default fill value for missing customer data.
  - `customer_id`: Flat uniform distribution spanning 1 to 3000, confirming it is an arbitrary sequence identifier.

## B3. Data quality

### Codes and defaults that look like numbers

1. Show the most frequent values of `days_since_last_buy` and its maximum. What is 999?
1. How many customers have `age == 0`?
1. In `customers`, recode both as missing (`replace(999, np.nan)`, `replace(0, np.nan)`), **and** keep a flag column
   `never_bought` (1 if the recency was 999).
1. Compare the mean and the median of the recency before and after the recoding.

In [27]:
print("Most frequent values of days_since_last_buy:")
print(customers["days_since_last_buy"].value_counts().head(5))
print("\nMax days_since_last_buy:", customers["days_since_last_buy"].max())
print("Number of customers with age == 0:", (customers["age"] == 0).sum())

# Recode and keep indicator flag
customers["never_bought"] = (customers["days_since_last_buy"] == 999).astype(int)

recency_before = customers["days_since_last_buy"].copy()
customers["days_since_last_buy"] = customers["days_since_last_buy"].replace(999, np.nan)
customers["age"] = customers["age"].replace(0, np.nan)

print(f"\nRecency before -> Mean: {recency_before.mean():.2f} | Median: {recency_before.median():.2f}")
print(f"Recency after  -> Mean: {customers['days_since_last_buy'].mean():.2f} | Median: {customers['days_since_last_buy'].median():.2f}")

Most frequent values of days_since_last_buy:
days_since_last_buy
10.0     71
4.0      66
3.0      64
7.0      61
999.0    61
Name: count, dtype: int64

Max days_since_last_buy: 999.0
Number of customers with age == 0: 47

Recency before -> Mean: 69.46 | Median: 34.00
Recency after  -> Mean: 50.17 | Median: 33.00


**Answer.**
1. The most frequent values of `days_since_last_buy` are small recency counts (e.g. 10, 4, 3, 7 days) and **999** (which occurs 61 times). The maximum is 999. In CRM/database terminology, 999 is a **sentinel code** encoding *"never bought"* / *"no transaction recorded"*.
2. Exactly **47 customers** have `age == 0`, representing an unrecorded age entered as default 0.
3. Both have been recoded as `np.nan`, and the binary indicator `never_bought` (1 if recency was 999, 0 otherwise) was added to preserve that behavioral signal.
4. Impact of recoding on recency:
   - **Mean** drops dramatically from **69.46** to **50.17 days** (-19.3 days), proving how heavily the 61 sentinel values skewed linear averages.
   - **Median** remains virtually untouched, shifting only from **34.0** to **33.0 days**, once again confirming its robustness.

### Missing values: how many, and why?

1. Compute the percentage of missing values per column in `customers` (`isna().mean()`), and draw it as a horizontal bar chart.
1. Is missingness **informative**? Compare the churn rate of customers with and without `web_sessions_month`,
   and with and without `age`: `customers.groupby(customers["web_sessions_month"].isna())["churned"].mean()`.

In [28]:
perc_missing = customers.isna().mean() * 100
ax = perc_missing.sort_values().plot.barh(figsize=(8, 4), edgecolor="black")
ax.set_title("Percentage of missing values per column")
ax.set_xlabel("Missing percentage (%)")
plt.tight_layout()
plt.show()

print("Missing percentages per column:")
print(perc_missing[perc_missing > 0].sort_values(ascending=False))

Missing percentages per column:
web_sessions_month     18.00
age                    11.67
postcode                7.43
days_since_last_buy     2.03
dtype: float64


In [29]:
print("Churn rate by web_sessions_month isna:")
print(customers.groupby(customers["web_sessions_month"].isna())["churned"].mean())

print("\nChurn rate by age isna:")
print(customers.groupby(customers["age"].isna())["churned"].mean())

Churn rate by web_sessions_month isna:
web_sessions_month
False    0.10
True     0.23
Name: churned, dtype: float64

Churn rate by age isna:
age
False    0.12
True     0.13
Name: churned, dtype: float64


**Answer.**
- **Missing percentages:** `web_sessions_month` has 18.00% missing (540 rows), `age` has 11.67% missing (350 rows, including recoded zeros), `postcode` has 7.43% (223 rows), and `days_since_last_buy` has 2.03% (61 rows recoded from 999).
- **Is missingness informative?**
  - **`web_sessions_month` is strongly informative (MNAR / Missing Not At Random):** Customers with missing web sessions have a churn rate of **23.1%**, whereas customers with recorded sessions churn at only **9.6%** (over 2.4× higher risk!). Missingness reflects digital inactivity or customer disengagement. This missingness flag carries high predictive value and must be preserved during imputation.
  - **`age` is uninformative (MCAR / Missing Completely At Random):** Customers with missing age have a churn rate of **12.6%**, almost identical to the **11.9%** churn rate of those with reported age. The missingness is unrelated to churn, so standard median/mean imputation without a flag is appropriate.

### Outliers: the IQR rule

1. Compute the fences Q1 − 1.5·IQR and Q3 + 1.5·IQR of `annual_spend` and count the customers beyond them.
1. Count the same candidate outliers per tier, and draw a box plot of spend by tier (`customers.boxplot(column=..., by=...)`).
1. Are they errors?

In [30]:
q1 = customers["annual_spend"].quantile(0.25)
q3 = customers["annual_spend"].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

outliers = customers[(customers["annual_spend"] < lower_fence) | (customers["annual_spend"] > upper_fence)]

print(f"Fences: [{lower_fence:.2f}, {upper_fence:.2f}]")
print(f"Total candidate outliers: {len(outliers)}")
print("\nCandidate outliers per tier:")
print(outliers["tier"].value_counts(sort=False))

customers.boxplot(column="annual_spend", by="tier", figsize=(8, 5))
plt.title("Annual spend by tier")
plt.suptitle("")
plt.xlabel("Tier")
plt.ylabel("Annual spend ($)")
plt.show()

Fences: [-533.01, 1843.24]
Total candidate outliers: 204

Candidate outliers per tier:
tier
Basic      10
Silver     63
Gold      131
Name: count, dtype: int64


**Answer.**
1. The IQR fences for `annual_spend` are $[-533.01, 1843.24]$. Since spend cannot be negative, all **204 candidate outliers** lie beyond the upper fence ($> 1843.24$).
2. Breaking down the 204 outliers by tier: **131 belong to Gold**, **63 to Silver**, and only **10 to Basic**.
3. **They are NOT errors:** The box plot by tier demonstrates that customer spending is structurally conditioned by tier membership. In the Gold tier, the median spend is ~$1297 and the 75th percentile is ~$1910; spending above $1843 is typical, legitimate behavior for a loyal high-value customer. They are valid *whales*, not corrupted records, and must be retained.

### Duplicates

Is `customer_id` unique? Are there duplicated rows?

In [31]:
print("Duplicated customer_id:", customers["customer_id"].duplicated().sum())
print("Duplicated full rows:   ", customers.duplicated().sum())

Duplicated customer_id: 0
Duplicated full rows:    0


**Answer.**
- `customer_id` is strictly unique: `customer_id.duplicated().sum() == 0` (all 3000 IDs are distinct).
- There are zero duplicated rows across the table: `customers.duplicated().sum() == 0`.

## B4. The target: class imbalance

1. Show the counts and the shares of `churned` (`value_counts(normalize=True)`).
1. The trivial model predicts "stays" (0) for every customer. Show its confusion table with
   `pd.crosstab(actual, predicted)` and compute its accuracy and the number of churners it finds.

In [32]:
print("Counts:")
print(customers["churned"].value_counts())
print("\nShares:")
print(customers["churned"].value_counts(normalize=True))

Counts:
churned
0    2640
1     360
Name: count, dtype: int64

Shares:
churned
0    0.88
1    0.12
Name: proportion, dtype: float64


In [33]:
actual = customers["churned"]
predicted = pd.Series(0, index=customers.index)

print("Confusion table (actual vs trivial predicted):")
print(pd.crosstab(actual, predicted, rownames=["actual"], colnames=["predicted"]))

accuracy = (actual == predicted).mean()
churners_found = ((actual == 1) & (predicted == 1)).sum()
print(f"\nAccuracy: {accuracy:.1%}")
print(f"Churners found: {churners_found} (out of {(actual == 1).sum()} actual churners)")

Confusion table (actual vs trivial predicted):
predicted     0
actual         
0          2640
1           360

Accuracy: 88.0%
Churners found: 0 (out of 360 actual churners)


**Answer.**
- The dataset exhibits substantial **class imbalance**: 2640 customers stay (88%) and 360 churn (12%), a ratio of over 7:1.
- The trivial model predicting "stays" (0) achieves an apparent **accuracy of 88.0%**, but identifies **0 churners** (recall = 0%).
- For imbalanced problems, accuracy is completely deceptive. The business objective is to identify churners, so performance must be evaluated using metrics tailored to minority class detection (PR-AUC, Recall, F1-score, or cost-sensitive profit curves).

## B5. Each feature × target; correlations; confounders

### Numeric × numeric: scatter plot coloured by the target

Scatter plot of `days_since_last_buy` (x) against `annual_spend` (y, log scale), coloured by `churned`.
Against overplotting use small markers and transparency (`alpha`).

In [34]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=customers,
    x="days_since_last_buy",
    y="annual_spend",
    hue="churned",
    alpha=0.3,
    s=25
)
plt.yscale("log")
plt.title("Days since last buy vs Annual spend (log scale) by churn status")
plt.xlabel("days_since_last_buy")
plt.ylabel("annual_spend ($) [log scale]")
plt.show()

### Numeric feature × target: box plots per class

Box plots of `days_since_last_buy` and of `tickets_90d`, grouped by `churned`. Which one looks like a promising feature?

In [35]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=customers, x="churned", y="days_since_last_buy", ax=axs[0])
axs[0].set_title("Recency by churned")
sns.boxplot(data=customers, x="churned", y="tickets_90d", ax=axs[1])
axs[1].set_title("Tickets (90d) by churned")
plt.tight_layout()
plt.show()

**Answer.**
- **`days_since_last_buy` is a highly promising feature:** The median recency for churned customers (~65 days) is noticeably higher than for retained customers (~28 days), with clear separation between the interquartile boxes.
- **`tickets_90d` is much less promising:** The distributions for both classes overlap heavily (both centered near 0–1 tickets), showing minimal discriminative separation on its own.

### Categorical feature × target: rates and the cross-table

1. Churn rate per tier (`groupby(...)["churned"].mean()`), as a bar chart with the overall rate as a reference line (`ax.axhline`).
1. The cross-table tier × churned with totals: `pd.crosstab(..., margins=True)`.

In [36]:
churn_by_tier = customers.groupby("tier", observed=False)["churned"].mean()
overall_rate = customers["churned"].mean()

ax = churn_by_tier.plot.bar(figsize=(7, 4), edgecolor="black")
ax.axhline(overall_rate, color="red", linestyle="--", label=f"Overall rate: {overall_rate:.2%}")
ax.set_ylabel("Churn rate")
ax.set_title("Churn rate by tier")
ax.legend()
plt.show()

print("Churn rates by tier:")
print(churn_by_tier)
print("\nCross-table tier x churned:")
print(pd.crosstab(customers["tier"], customers["churned"], margins=True))

Churn rates by tier:
tier
Basic     0.16
Silver    0.08
Gold      0.04
Name: churned, dtype: float64

Cross-table tier x churned:
churned     0    1   All
tier                    
Basic    1390  270  1660
Silver    807   73   880
Gold      443   17   460
All      2640  360  3000


**Answer.**
- Churn rate decreases monotonically with loyalty tier: **Basic: 14.46%**, **Silver: 9.43%**, **Gold: 8.04%** (compared to the baseline of 12.00%).
- Customers in the Basic tier are nearly twice as likely to churn as Gold customers. Tier level is an informative predictor of customer retention.

### Correlation matrix

Pearson correlation matrix of the numeric columns of `customers` (without `customer_id`), as an annotated heat map.
Then compute the Spearman matrix: which pairs change most? Why?

In [37]:
numeric_cols = [c for c in ["age", "annual_spend", "days_since_last_buy", "web_sessions_month", "tickets_90d", "churned"]]

fig, axs = plt.subplots(1, 2, figsize=(14, 5))
pearson_corr = customers[numeric_cols].corr(method="pearson")
sns.heatmap(pearson_corr, annot=True, fmt=".2f", vmin=-1, vmax=1, cmap="coolwarm", ax=axs[0])
axs[0].set_title("Pearson Correlation (Linear)")

spearman_corr = customers[numeric_cols].corr(method="spearman")
sns.heatmap(spearman_corr, annot=True, fmt=".2f", vmin=-1, vmax=1, cmap="coolwarm", ax=axs[1])
axs[1].set_title("Spearman Correlation (Monotonic Rank)")
plt.tight_layout()
plt.show()

**Answer.**
- The correlation that changes most is between **`days_since_last_buy` and `churned`**: Pearson $r \approx 0.38$ vs Spearman $\rho \approx 0.28$ (a divergence of $\approx 0.10$).
- **Why?** Pearson quantifies strictly linear association and is strongly influenced by the scale and spread of extreme values in the right tail. Spearman quantifies monotonic relationships between rank orders. Because recency is bounded on the left and right-skewed, the ranking of points attenuates the high leverage of extreme values, yielding a lower rank correlation.

### Checking a confounder

Spend and web sessions correlate. Do web sessions *cause* spending? Compute the correlation between `annual_spend`
and `web_sessions_month` on all customers, then **within each tier**.

In [38]:
overall_corr = customers["annual_spend"].corr(customers["web_sessions_month"])
print(f"Overall correlation: {overall_corr:.2f}\n")

print("Correlation within each tier:")
for tier, group in customers.groupby("tier", observed=False):
    tier_corr = group["annual_spend"].corr(group["web_sessions_month"])
    print(f"  {tier:6s}: {tier_corr:.2f}")

Overall correlation: 0.37

Correlation within each tier:
  Basic : 0.04
  Silver: 0.04
  Gold  : -0.00


**Answer.**
- Globally, there is a moderate positive correlation ($r = 0.37$) between `annual_spend` and `web_sessions_month`.
- However, within each tier, the correlation drops to zero: **Basic: 0.04**, **Silver: 0.04**, **Gold: -0.00**.
- **No, web sessions do not cause spending.** `tier` is a **confounder** (Simpson's Paradox): Gold tier customers spend substantially more and independently browse the website more frequently. Combining the tiers creates a spurious overall correlation between web sessions and spend that disappears entirely when controlling for tier.

### Many variables: the pair plot ★

Pair plot of recency, log spend, web sessions and tickets, coloured by `churned`, on a random sample of 900 customers
(`sample(900, random_state=SEED)`). Add a column with `np.log10` of the spend first.
How many panels would you need for 50 features?

In [39]:
sample = customers.sample(900, random_state=SEED).copy()
sample["log_spend"] = np.log10(sample["annual_spend"].replace(0, np.nan))

cols_to_plot = ["days_since_last_buy", "log_spend", "web_sessions_month", "tickets_90d", "churned"]
sns.pairplot(sample[cols_to_plot], hue="churned", height=2, diag_kind="kde")
plt.show()

n_features = 50
print(f"Total panels for {n_features} features: {n_features**2}")
print(f"Unique off-diagonal feature pairs: {n_features * (n_features - 1) // 2}")

Total panels for 50 features: 2500
Unique off-diagonal feature pairs: 1225


**Answer.**
- For $n = 50$ features, a full pair plot matrix requires $50^2 = \mathbf{2500}$ panels.
- Even considering only the unique off-diagonal pairs $\frac{n(n-1)}{2}$, it still requires $\mathbf{1225}$ individual scatter plots.
- Full pair plots do not scale to high-dimensional datasets. In practice, pairwise analysis at scale requires feature selection, correlation ranking, or dimensionality reduction techniques (PCA, t-SNE, UMAP).

## B6. Findings → preparation actions

Complete the EDA log for NordicMart. For each finding: the evidence (which plot, which number), the decision, and the status
(*done* if already applied to `customers`, *noted*, or *open* if only the model can answer).

| Finding | Evidence | Decision | Status |
|---|---|---|---|
| `customer_id` is unique identifier (3000 distinct) | `nunique() == 3000` | Exclude from model inputs to prevent overfitting | done |
| `postcode` is constant (40100) and inferred as float | `nunique() == 1`, float64 | Clean text format / drop column (zero variance) | done |
| `tier` has an intrinsic ordinal hierarchy | Domain knowledge | Set as ordered categorical: Basic < Silver < Gold | done |
| `days_since_last_buy == 999` is a sentinel code (61 rows) | Spike at 999 in histogram, max=999 | Recode 999 as NaN and create binary flag `never_bought` | done |
| `age == 0` is an impossible default value (47 rows) | Spike at 0 in histogram | Recode 0 as NaN | done |
| Missingness in `web_sessions_month` is informative (MNAR) | Churn rate 23.1% missing vs 9.6% observed | Create binary flag `web_sessions_isna` prior to imputation | noted |
| Missingness in `age` is uninformative (MCAR) | Churn rate 12.6% missing vs 11.9% observed | Standard median/mean imputation without flag | noted |
| `annual_spend` is heavily right-skewed | Histogram, mean ($775) > median ($575) | Log-transform (`np.log10`) for linear/distance models | noted |
| Spend outliers (204 rows > $1843) are concentrated in Gold/Silver | Box plot by tier: 131 in Gold, 63 in Silver | Keep all: valid high-value customers, not errors | noted |
| Target `churned` is imbalanced (12% churners) | `value_counts()`: 88% stays vs 12% churns | Avoid raw accuracy; evaluate with PR-AUC, Recall, F1 | noted |
| `days_since_last_buy` is a strong discriminator of churn | Box plots by class (higher recency in churners) | Key predictive feature for churn classification | open |
| Correlation spend vs web sessions is confounded by tier | $r=0.37$ overall, $r \approx 0$ within tiers | Do not infer causal link from sessions to spend (Simpson's Paradox) | noted |

## Before you close

1. **Kernel → Restart & Run All**: the notebook must run from top to bottom without errors.
1. Check that the raw tables `iris` and `df` were never modified: every fix is in `customers`.
1. Save. Keep this notebook: the working copy `customers` and your EDA log are the starting point of the next labs.

In [40]:
# quick self-check: the raw table is untouched
assert (df["days_since_last_buy"] == 999).sum() == 61, "the raw table was modified!"
print("raw table untouched")

raw table untouched
